# PINN Inverse Bioheat Solver for 3D Breast Thermography
Solves the inverse Pennes Bioheat equation on reconstructed 3D breast geometries
to estimate tumour location, size, and metabolic heat generation from surface thermography.

**Upstream dependency:** `breastnet3d_v4.ipynb` (watertight `.stl` + raw `.tiff` data)

## 1. Install Dependencies

In [ ]:
!pip install trimesh torch numpy scipy tifffile opencv-python matplotlib tqdm plotly

## 2. Imports & Biophysical Constants

In [ ]:
import sys, os, math, struct
import numpy as np
import torch
import torch.nn as nn
import trimesh
import tifffile, cv2
from pathlib import Path
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from tqdm.auto import tqdm

BREASTNET_DIR = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\UNET_Segmentation\3DBreastnet"
sys.path.insert(0, BREASTNET_DIR)
from models import UNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

K_TISSUE  = 0.48
WB        = 0.0005
CB        = 3600.0
TA        = 37.0
QM        = 450.0
COORD_SCALE = 1e-3

UNET_CKPT   = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\UNET_Segmentation\breast_segmentation_unet_best_gpu.pth"
TIFF_BASE   = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\organized_by_patient"
STL_DIR     = os.path.join(BREASTNET_DIR, "exported_stls")
CKPT_3D     = os.path.join(BREASTNET_DIR, "checkpoints_3d", "3dbreastnet_best.pth")

## 3. STL Mesh Loader (Unit-Corrected)

In [ ]:
def load_mesh_mm(stl_path):
    mesh = trimesh.load(stl_path)
    extents = mesh.bounding_box.extents
    if extents.max() < 1.0:
        mesh.apply_scale(1000.0)
    elif extents.max() < 10.0:
        mesh.apply_scale(100.0)
    trimesh.repair.fix_normals(mesh)
    print(f"  Extents (mm): {mesh.bounding_box.extents.round(1)}")
    print(f"  Watertight: {mesh.is_watertight}  |  Volume: {mesh.volume:.1f} mm3")
    return mesh

## 4. Surface Temperature Projection (Mask-Gated)

In [ ]:
def load_unet():
    unet = UNet().to(DEVICE)
    unet.load_state_dict(torch.load(UNET_CKPT, map_location=DEVICE, weights_only=False))
    unet.eval()
    return unet

def get_masked_thermal(unet, tiff_path):
    raw = tifffile.imread(str(tiff_path)).astype(np.float32)
    mn, mx = raw.min(), raw.max()
    norm = (raw - mn) / (mx - mn + 1e-8)
    with torch.no_grad():
        inp = torch.tensor(norm).unsqueeze(0).unsqueeze(0).to(DEVICE)
        mask = (torch.sigmoid(unet(inp)).squeeze().cpu().numpy() > 0.5)
    mask = binary_erosion(mask, iterations=2)
    result = raw.copy()
    result[~mask] = np.nan
    return result

def get_view_key(fn):
    n = fn.lower()
    if "right later" in n: return "RL"
    if "right obli"  in n: return "RO"
    if "frontal" in n or "anterior" in n: return "F"
    if "left obliq"  in n: return "LO"
    if "left later"  in n: return "LL"
    return None

VIEW_ANGLES_DEG = {"RL": -90., "RO": -45., "F": 0., "LO": 45., "LL": 90.}

def project_thermal_onto_mesh(unet, mesh, patient_dir):
    verts = np.array(mesh.vertices)
    normals = np.array(mesh.vertex_normals)
    N = len(verts)
    bb_min = verts.min(axis=0)
    bb_max = verts.max(axis=0)
    bb_center = (bb_min + bb_max) / 2.0
    bb_half   = (bb_max - bb_min) / 2.0
    verts_norm = (verts - bb_center) / bb_half
    temp_sum   = np.zeros(N)
    temp_count = np.zeros(N)

    label_dirs = [d for d in Path(patient_dir).iterdir() if d.is_dir()]
    if not label_dirs:
        print(f"  No label directory found in {patient_dir}")
        return None, None, None
    tiff_dir = label_dirs[0]

    for tiff_path in tiff_dir.glob("*.tiff"):
        vk = get_view_key(tiff_path.name)
        if vk is None: continue
        angle_deg = VIEW_ANGLES_DEG[vk]
        rad = angle_deg * math.pi / 180.0
        c, s = math.cos(rad), math.sin(rad)
        Vx = verts_norm[:, 0]; Vy = verts_norm[:, 1]; Vz = verts_norm[:, 2]
        X_cam =  c * Vx + s * Vz
        Y_cam =  Vy
        NZ_cam = -s * normals[:, 0] + c * normals[:, 2]
        visible = NZ_cam < 0
        thermal = get_masked_thermal(unet, tiff_path)
        H, W = thermal.shape
        px = np.clip(((X_cam + 1.0) / 2.0) * (W - 1), 0, W - 1).astype(int)
        py = np.clip(((Y_cam + 1.0) / 2.0) * (H - 1), 0, H - 1).astype(int)
        for vi in range(N):
            if visible[vi]:
                t = thermal[py[vi], px[vi]]
                if not np.isnan(t):
                    temp_sum[vi] += t
                    temp_count[vi] += 1

    projected = temp_count > 0
    T_measured = np.full(N, np.nan)
    T_measured[projected] = temp_sum[projected] / temp_count[projected]
    confidence = np.where(projected, 1.0, 0.0)
    if (~projected).any() and projected.any():
        tree = cKDTree(verts[projected])
        _, idx = tree.query(verts[~projected], k=5)
        T_measured[~projected] = np.mean(T_measured[projected][idx], axis=1)
        confidence[~projected] = 0.3
    print(f"  Projected: {projected.sum()}/{N} vertices | KNN filled: {(~projected).sum()}")
    print(f"  Temp range: {np.nanmin(T_measured):.1f} C - {np.nanmax(T_measured):.1f} C")
    return verts, T_measured, confidence

## 5. PINN Architecture

In [ ]:
class BioheatPINN(nn.Module):
    def __init__(self, hidden=256, depth=6):
        super().__init__()
        layers = [nn.Linear(3, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)
        self.x_t   = nn.Parameter(torch.tensor([0.0]))
        self.y_t   = nn.Parameter(torch.tensor([0.0]))
        self.z_t   = nn.Parameter(torch.tensor([0.0]))
        self.r_t   = nn.Parameter(torch.tensor([10.0]))
        self.Q_max = nn.Parameter(torch.tensor([5000.0]))

    def forward(self, xyz):
        return self.net(xyz).squeeze(-1)

    def Q_tumor(self, xyz):
        d2 = ((xyz[:, 0] - self.x_t)**2 +
              (xyz[:, 1] - self.y_t)**2 +
              (xyz[:, 2] - self.z_t)**2)
        return self.Q_max * torch.exp(-d2 / (self.r_t**2 + 1e-8))

## 6. PDE Residual (Pennes Bioheat)

In [ ]:
def laplacian(T, xyz):
    dT = torch.autograd.grad(
        T, xyz, grad_outputs=torch.ones_like(T), create_graph=True
    )[0]
    lap = torch.zeros_like(T)
    for i in range(3):
        d2T = torch.autograd.grad(
            dT[:, i], xyz,
            grad_outputs=torch.ones_like(dT[:, i]),
            create_graph=True
        )[0][:, i]
        lap = lap + d2T
    return lap

def pde_residual(model, xyz_norm, bbox_extents_mm):
    T   = model(xyz_norm)
    lap = laplacian(T, xyz_norm)
    scale_factor = (COORD_SCALE * bbox_extents_mm / 2.0) ** 2
    avg_scale = scale_factor.mean()
    lap_physical = lap / avg_scale
    Q_t = model.Q_tumor(xyz_norm)
    residual = K_TISSUE * lap_physical + WB * CB * (TA - T) + QM + Q_t
    return residual

## 7. PINN Training Loop

In [ ]:
def train_pinn(model, surface_pts, T_measured, confidence,
               interior_pts, bbox_extents_mm,
               adam_steps=10000, lbfgs_steps=500, device=DEVICE):
    model = model.to(device)
    s_pts = torch.tensor(surface_pts, dtype=torch.float32, device=device)
    s_T   = torch.tensor(T_measured, dtype=torch.float32, device=device)
    s_w   = torch.tensor(confidence, dtype=torch.float32, device=device)
    bbox  = torch.tensor(bbox_extents_mm, dtype=torch.float32, device=device)
    i_pts = torch.tensor(interior_pts, dtype=torch.float32, device=device)

    all_pts = np.vstack([surface_pts, interior_pts])
    bb_min = all_pts.min(axis=0)
    bb_max = all_pts.max(axis=0)
    bb_center = torch.tensor((bb_min + bb_max) / 2, dtype=torch.float32, device=device)
    bb_half   = torch.tensor((bb_max - bb_min) / 2 + 1e-8, dtype=torch.float32, device=device)
    s_norm = (s_pts - bb_center) / bb_half
    i_norm_base = (i_pts - bb_center) / bb_half

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    lambda_pde = None
    losses_data, losses_pde = [], []
    print(f"Phase 1: Adam ({adam_steps} steps)")

    for step in range(adam_steps):
        optimizer.zero_grad()
        idx_s = torch.randperm(len(s_norm), device=device)[:2000]
        idx_i = torch.randperm(len(i_norm_base), device=device)[:5000]
        xyz_s = s_norm[idx_s]
        xyz_i = i_norm_base[idx_i].clone().requires_grad_(True)
        T_pred = model(xyz_s)
        L_data = (s_w[idx_s] * (T_pred - s_T[idx_s])**2).mean()
        f = pde_residual(model, xyz_i, bbox)
        L_pde = (f**2).mean()
        if lambda_pde is None:
            lambda_pde = (L_data / (L_pde + 1e-8)).detach()
            print(f"  lambda_pde auto-calibrated to {lambda_pde.item():.4e}")
        loss = L_data + lambda_pde * L_pde
        loss.backward()
        optimizer.step()
        model.r_t.data.clamp_(min=2.0)
        if step % 1000 == 0 or step == adam_steps - 1:
            print(f"  [{step:5d}] L_data={L_data.item():.4f}  L_pde={L_pde.item():.4f}  "
                  f"total={loss.item():.4f}  |  "
                  f"tumor=({model.x_t.item():.2f},{model.y_t.item():.2f},{model.z_t.item():.2f}) "
                  f"r={model.r_t.item():.2f}mm  Q={model.Q_max.item():.0f}")
        losses_data.append(L_data.item())
        losses_pde.append(L_pde.item())

    print(f"\nPhase 2: L-BFGS ({lbfgs_steps} steps)")
    lbfgs = torch.optim.LBFGS(model.parameters(), lr=0.1, max_iter=1,
                                line_search_fn="strong_wolfe")
    for step in range(lbfgs_steps):
        def closure():
            lbfgs.zero_grad()
            idx_s = torch.randperm(len(s_norm), device=device)[:2000]
            idx_i = torch.randperm(len(i_norm_base), device=device)[:5000]
            xyz_s = s_norm[idx_s]
            xyz_i = i_norm_base[idx_i].clone().requires_grad_(True)
            T_pred = model(xyz_s)
            L_data = (s_w[idx_s] * (T_pred - s_T[idx_s])**2).mean()
            f = pde_residual(model, xyz_i, bbox)
            L_pde = (f**2).mean()
            loss = L_data + lambda_pde * L_pde
            loss.backward()
            model.r_t.data.clamp_(min=2.0)
            return loss
        lbfgs.step(closure)
        if step % 100 == 0:
            print(f"  [{step:4d}] tumor=({model.x_t.item():.3f},{model.y_t.item():.3f},{model.z_t.item():.3f}) "
                  f"r={model.r_t.item():.2f}mm  Q={model.Q_max.item():.0f}")

    tumor_norm = torch.tensor([model.x_t.item(), model.y_t.item(), model.z_t.item()], device=device)
    tumor_mm = tumor_norm * bb_half + bb_center
    results = {
        "x_t_mm": tumor_mm[0].item(), "y_t_mm": tumor_mm[1].item(), "z_t_mm": tumor_mm[2].item(),
        "r_t_mm": model.r_t.item(), "Q_max_Wm3": model.Q_max.item(),
        "volume_mm3": (4/3) * math.pi * model.r_t.item()**3,
        "losses_data": losses_data, "losses_pde": losses_pde,
    }
    print(f"\n{'='*50}")
    print(f"RESULTS:")
    print(f"  Tumour centre: ({results['x_t_mm']:.2f}, {results['y_t_mm']:.2f}, {results['z_t_mm']:.2f}) mm")
    print(f"  Tumour radius: {results['r_t_mm']:.2f} mm")
    print(f"  Tumour volume: {results['volume_mm3']:.1f} mm3")
    print(f"  Q_max: {results['Q_max_Wm3']:.0f} W/m3")
    return results

## 8. 3D Visualisation (Plotly)

In [ ]:
def visualize_results(mesh, T_measured, results):
    verts = np.array(mesh.vertices)
    faces = np.array(mesh.faces)
    t_min = np.nanpercentile(T_measured, 2)
    t_max = np.nanpercentile(T_measured, 98)
    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        intensity=T_measured, colorscale='Inferno',
        cmin=t_min, cmax=t_max,
        showscale=True, colorbar_title="C",
        name="Surface Temperature", opacity=0.7
    ))
    r = results["r_t_mm"]
    cx, cy, cz = results["x_t_mm"], results["y_t_mm"], results["z_t_mm"]
    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 30)
    sx = cx + r * np.outer(np.cos(u), np.sin(v))
    sy = cy + r * np.outer(np.sin(u), np.sin(v))
    sz = cz + r * np.outer(np.ones_like(u), np.cos(v))
    fig.add_trace(go.Surface(
        x=sx, y=sy, z=sz,
        colorscale=[[0, 'red'], [1, 'red']],
        showscale=False, opacity=0.5, name="Estimated Tumour"
    ))
    fig.update_layout(
        title=f"Inverse Bioheat: Tumour at ({cx:.1f}, {cy:.1f}, {cz:.1f}) mm, r={r:.1f} mm",
        scene=dict(xaxis_title='X (mm)', yaxis_title='Y (mm)', zaxis_title='Z (mm)',
                   aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

def plot_losses(results):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.semilogy(results["losses_data"]); ax1.set_title("Data Loss"); ax1.set_xlabel("Step")
    ax2.semilogy(results["losses_pde"]);  ax2.set_title("PDE Loss");  ax2.set_xlabel("Step")
    plt.tight_layout(); plt.show()

## 9. Run: Single Patient Test

In [ ]:
TARGET_PATIENT = "Patient_37"

stl_path = os.path.join(STL_DIR, f"{TARGET_PATIENT}_3d_geometry.stl")
print(f"Loading mesh: {stl_path}")
mesh = load_mesh_mm(stl_path)

unet = load_unet()
patient_dir = os.path.join(TIFF_BASE, TARGET_PATIENT)
print(f"\nProjecting thermal data from {patient_dir}...")
surface_pts, T_measured, confidence = project_thermal_onto_mesh(unet, mesh, patient_dir)

print("\nSampling interior collocation points...")
interior_pts = trimesh.sample.volume_mesh(mesh, count=10000)
print(f"  Sampled {len(interior_pts)} interior points")

bbox_extents = mesh.bounding_box.extents
model = BioheatPINN(hidden=256, depth=6)
print(f"\nPINN parameters: {sum(p.numel() for p in model.parameters()):,}")

results = train_pinn(
    model, surface_pts, T_measured, confidence,
    interior_pts, bbox_extents,
    adam_steps=5000, lbfgs_steps=200
)

plot_losses(results)
visualize_results(mesh, T_measured, results)

## 10. Batch Processing (All Patients)

In [ ]:
def run_all_patients(adam_steps=5000, lbfgs_steps=200):
    unet = load_unet()
    stl_dir = Path(STL_DIR)
    all_results = []
    stl_files = sorted(stl_dir.glob("*_3d_geometry.stl"))
    print(f"Found {len(stl_files)} STL files\n")
    for stl_path in tqdm(stl_files):
        pid = stl_path.stem.replace("_3d_geometry", "")
        patient_dir = os.path.join(TIFF_BASE, pid)
        if not os.path.isdir(patient_dir):
            print(f"  Skip {pid}: no TIFF directory"); continue
        print(f"\n{'='*50}\nPatient: {pid}")
        try:
            mesh = load_mesh_mm(str(stl_path))
            if not mesh.is_watertight:
                print(f"  Skip {pid}: mesh not watertight"); continue
            surface_pts, T_measured, confidence = project_thermal_onto_mesh(unet, mesh, patient_dir)
            if T_measured is None: continue
            interior_pts = trimesh.sample.volume_mesh(mesh, count=10000)
            model = BioheatPINN()
            results = train_pinn(model, surface_pts, T_measured, confidence,
                                 interior_pts, mesh.bounding_box.extents,
                                 adam_steps=adam_steps, lbfgs_steps=lbfgs_steps)
            results["patient_id"] = pid
            all_results.append(results)
        except Exception as e:
            print(f"  Error on {pid}: {e}")
    import pandas as pd
    if all_results:
        df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ("losses_data", "losses_pde")}
                           for r in all_results])
        csv_path = "pinn_results_all_patients.csv"
        df.to_csv(csv_path, index=False)
        print(f"\nSaved results to {csv_path}")
        display(df)
    return all_results

# Uncomment to run:
# all_results = run_all_patients()